# Practical Exam: Customer Purchase Prediction

RetailTech Solutions is a fast-growing international e-commerce platform operating in over 20 countries across Europe, North America, and Asia. They specialize in fashion, electronics, and home goods, with a unique business model that combines traditional retail with a marketplace for independent sellers.

The company has seen rapid growth. A key part of their success has been their data-driven approach to personalization. However, as they plan their expansion into new markets, they need to improve their ability to predict customer behavior.

Their marketing team wants to predict which customers are most likely to make a purchase based on their browsing behavior.

As an AI Engineer, you will help build this prediction system. Your work will directly impact RetailTech's growth strategy and their goal of increasing revenue.


## Data Description

| Column Name | Criteria |
|------------|----------|
| customer_id | Integer. Unique identifier for each customer. No missing values. |
| time_spent | Float. Minutes spent on website per session. Missing values should be replaced with median. |
| pages_viewed | Integer. Number of pages viewed in session. Missing values should be replaced with mean. |
| basket_value | Float. Value of items in basket. Missing values should be replaced with 0. |
| device_type | String. One of: Mobile, Desktop, Tablet. Missing values should be replaced with "Unknown". |
| customer_type | String. One of: New, Returning. Missing values should be replaced with "New". |
| purchase | Binary. Whether customer made a purchase (1) or not (0). Target variable. |

# Task 1

The marketing team has collected customer session data in `raw_customer_data.csv`, but it contains missing values and inconsistencies that need to be addressed.
Create a cleaned version of the dataframe:

- Start with the data in the file `raw_customer_data.csv`
- Your output should be a DataFrame named `clean_data`
- All column names and values should match the table below.
</br>

| Column Name | Criteria |
|------------|----------|
| customer_id | Integer. Unique identifier for each customer. No missing values. |
| time_spent | Float. Minutes spent on website per session. Missing values should be replaced with median. |
| pages_viewed | Integer. Number of pages viewed in session. Missing values should be replaced with mean. |
| basket_value | Float. Value of items in basket. Missing values should be replaced with 0. |
| device_type | String. One of: Mobile, Desktop, Tablet. Missing values should be replaced with "Unknown". |
| customer_type | String. One of: New, Returning. Missing values should be replaced with "New". |
| purchase | Binary. Whether customer made a purchase (1) or not (0). Target variable. |

In [4]:
import pandas as pd
import numpy as np

clean_data = pd.read_csv("raw_customer_data.csv")

clean_data["customer_id"] = pd.to_numeric(clean_data["customer_id"], errors="coerce").astype(int)

clean_data["time_spent"] = pd.to_numeric(clean_data["time_spent"], errors="coerce")
clean_data["time_spent"] = clean_data["time_spent"].fillna(clean_data["time_spent"].median()).astype(float)

clean_data["pages_viewed"] = pd.to_numeric(clean_data["pages_viewed"], errors="coerce")
clean_data["pages_viewed"] = clean_data["pages_viewed"].fillna(clean_data["pages_viewed"].mean()).round().astype(int)

clean_data["basket_value"] = pd.to_numeric(clean_data["basket_value"], errors="coerce")
clean_data["basket_value"] = clean_data["basket_value"].fillna(0).astype(float)

device = clean_data["device_type"].astype(str).str.strip().str.title()
clean_data["device_type"] = device.where(device.isin(["Mobile", "Desktop", "Tablet"]), "Unknown")

customer = clean_data["customer_type"].astype(str).str.strip().str.title()
clean_data["customer_type"] = customer.where(customer.isin(["New", "Returning"]), "New")

clean_data["purchase"] = pd.to_numeric(clean_data["purchase"], errors="coerce").astype(int)

clean_data = clean_data[["customer_id", "time_spent", "pages_viewed", "basket_value",
                         "device_type", "customer_type", "purchase"]]

print(clean_data.dtypes)
print(clean_data.isna().sum())
print(clean_data["device_type"].value_counts())
print(clean_data["customer_type"].value_counts())
clean_data.head()

customer_id        int64
time_spent       float64
pages_viewed       int64
basket_value     float64
device_type       object
customer_type     object
purchase           int64
dtype: object
customer_id      0
time_spent       0
pages_viewed     0
basket_value     0
device_type      0
customer_type    0
purchase         0
dtype: int64
Mobile     229
Desktop    170
Tablet      82
Unknown     19
Name: device_type, dtype: int64
Returning    291
New          209
Name: customer_type, dtype: int64


,customer_id,time_spent,pages_viewed,basket_value,device_type,customer_type,purchase
0,1,23.097867,7,50.574647,Mobile,Returning,0
1,2,57.092144,3,56.891022,Mobile,Returning,1
2,3,44.187643,14,8.348296,Mobile,Returning,0
3,4,36.320851,10,43.481489,Mobile,New,1
4,5,10.205100,16,0.000000,Mobile,Returning,1


# Task 2
The pre-cleaned dataset `model_data.csv` needs to be prepared for our neural network.
Create the model features:

- Start with the data in the file `model_data.csv`
- Scale numerical features (`time_spent`, `pages_viewed`, `basket_value`) to 0-1 range
- Apply one-hot encoding to the categorical features (`device_type`, `customer_type`)
    - The column names should have the following format: variable_name_category_name (e.g., `device_type_Desktop`)
- Your output should be a DataFrame named `model_feature_set`, with all column names from `model_data.csv` except for the columns where one-hot encoding was applied.


In [6]:
import pandas as pd

model_feature_set = pd.read_csv("model_data.csv")

for col in ["time_spent", "pages_viewed", "basket_value"]:
    col_min = model_feature_set[col].min()
    col_max = model_feature_set[col].max()
    model_feature_set[col] = (model_feature_set[col] - col_min) / (col_max - col_min)

model_feature_set = pd.get_dummies(model_feature_set, columns=["device_type", "customer_type"],
                                   prefix=["device_type", "customer_type"], prefix_sep="_", dtype=int)

print(model_feature_set.columns.tolist())
print(model_feature_set[["time_spent", "pages_viewed", "basket_value"]].agg(["min", "max"]))
model_feature_set.head()

['customer_id', 'time_spent', 'pages_viewed', 'basket_value', 'purchase', 'device_type_Desktop', 'device_type_Mobile', 'device_type_Tablet', 'device_type_Unknown', 'customer_type_New', 'customer_type_Returning']
     time_spent  pages_viewed  basket_value
min         0.0           0.0           0.0
max         1.0           1.0           1.0


,customer_id,time_spent,pages_viewed,basket_value,purchase,device_type_Desktop,device_type_Mobile,device_type_Tablet,device_type_Unknown,customer_type_New,customer_type_Returning
0,501,0.664167,0.500000,0.000000,1,1,0,0,0,1,0
1,502,0.483681,0.222222,0.524981,1,0,1,0,0,0,1
2,503,0.231359,0.111111,0.457291,0,0,1,0,0,0,1
3,504,0.792944,0.277778,0.000000,1,0,0,0,1,1,0
4,505,0.649210,0.166667,0.484283,1,0,0,1,0,1,0


# Task 3

Now that all preparatory work has been done, create and train a neural network that would allow the company to predict purchases.

- Using PyTorch, create a network with:
   - At least one hidden layer with 8 units
   - ReLU activation for hidden layer
   - Sigmoid activation for the output layer
- Using the prepared features in `input_model_features.csv`, train the model to predict purchases. 
- Use the validation dataset `validation_features.csv` to predict new values based on the trained model. 
- Your model should be named `purchase_model` and your output should be a DataFrame named `validation_predictions` with columns `customer_id` and `purchase`. The `purchase` column must be your predicted values.


In [5]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(42)
np.random.seed(42)

train_df = pd.read_csv("input_model_features.csv")
val_df = pd.read_csv("validation_features.csv")

feature_cols = [c for c in train_df.columns if c not in ["customer_id", "purchase"]]

X_train_df = train_df[feature_cols].astype(float)
X_val_df = val_df.reindex(columns=feature_cols, fill_value=0).astype(float)

col_min = X_train_df.min()
col_range = (X_train_df.max() - col_min).replace(0, 1)
X_train_df = (X_train_df - col_min) / col_range
X_val_df = (X_val_df - col_min) / col_range

X_train = torch.tensor(X_train_df.values, dtype=torch.float32)
y_train = torch.tensor(train_df["purchase"].values, dtype=torch.float32).view(-1, 1)
X_val = torch.tensor(X_val_df.values, dtype=torch.float32)

purchase_model = nn.Sequential(
    nn.Linear(X_train.shape[1], 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(purchase_model.parameters(), lr=0.01)

purchase_model.train()
for epoch in range(500):
    optimizer.zero_grad()
    loss = criterion(purchase_model(X_train), y_train)
    loss.backward()
    optimizer.step()

purchase_model.eval()
with torch.no_grad():
    train_acc = ((purchase_model(X_train) >= 0.5).float() == y_train).float().mean().item()
    val_pred = (purchase_model(X_val) >= 0.5).int().view(-1).numpy()

validation_predictions = pd.DataFrame({
    "customer_id": val_df["customer_id"].values,
    "purchase": val_pred
})

print("Final training loss:", round(loss.item(), 4))
print("Training accuracy:", round(train_acc, 4))
print(validation_predictions["purchase"].value_counts())
validation_predictions.head()

Final training loss: 0.4721
Training accuracy: 0.7875
1    197
0      3
Name: purchase, dtype: int64


,customer_id,purchase
0,1801,1
1,1802,1
2,1803,1
3,1804,0
4,1805,1
